# 4.2 — Weight Aligning (free)

> Run `4.0_cl_setup.ipynb` first.

**Mechanism:** after each task, rescale the new-class classifier rows so their mean norm
matches the old classes'.

**Memory:** 0 bytes. No data stored, no teacher kept.

## Why this arm is the one that matters

Most "forgetting" in a classifier is **task-recency bias** — the new classes' logits
simply grow larger, so they win the argmax regardless of the features. Weight Aligning
(Zhao et al., CVPR 2020) corrects that imbalance directly.

**This is the honest baseline for replay to beat.** If SmartReplay (4.5) only matches
WeightAlign, then ~960 KB/class bought you nothing that a free rescale doesn't give —
and that is a finding worth reporting.

## Setup

In [ ]:
# ── ULAL bootstrap — self-contained, no Drive dependency ────────────────────
# One shallow git clone instead of hundreds of API calls. subprocess takes a
# LIST, never a shell string, so the '&' in the branch name is safe.
import os, sys, subprocess
from getpass import getpass

REPO, BRANCH, ROOT = "silvergjeka22/Unified-Lifelong-Action-Learning", "GAN&ActiveLearning", "/content/ulal"
token = os.environ.get("GITHUB_TOKEN") or getpass("GitHub token (hidden): ")
os.environ["GITHUB_TOKEN"] = token

if os.path.isdir(f"{ROOT}/.git"):
    subprocess.run(["git", "-C", ROOT, "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", ROOT, "reset", "--hard", "FETCH_HEAD"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                    f"https://{token}@github.com/{REPO}.git", ROOT], check=True)

sys.path.insert(0, ROOT)
print(subprocess.run(["git", "-C", ROOT, "log", "-1", "--oneline"],
                     capture_output=True, text=True).stdout.strip())

In [ ]:
# ── Drive + dataset + paths ─────────────────────────────────────────────────
# Exports ULAL_* env vars that config.py reads. Nothing is written to disk and
# no kernel restart is needed.
os.environ["KAGGLE_USERNAME"] = "YOUR_KAGGLE_USER"
os.environ["KAGGLE_KEY"]      = "YOUR_KAGGLE_KEY"

from src.bootstrap import setup
setup(drive=True, dataset=True)

In [ ]:
%run /content/ulal/src/imports.py

In [ ]:
# ── Global label space (src/data/cache.py) ──────────────────────────────────
TASK_CLASSES = [cfg.SELECTED_CLASSES, cfg.TASK_1, cfg.TASK_2, cfg.TASK_4]
TASK_ROOTS   = [cfg.BASE_ROOT, cfg.TASK1_ROOT, cfg.TASK2_ROOT, cfg.TASK4_ROOT]
TASK_NAMES   = ["Base", "Task1", "Task2", "Task4"]
CL_TASKS     = [1, 2]     # Sections 4-5 stream over these
FEWSHOT_TASK = 3          # Section 7 probe only

global_classes, global_c2i, NUM_AFTER = build_label_space(TASK_CLASSES)
describe_label_space(TASK_CLASSES, TASK_NAMES, NUM_AFTER, task_3=cfg.TASK_3)

In [ ]:
# ── Load cached frame features (src/data/cache.py) ──────────────────────────
CACHE, LSTM_STATE, META = load_feature_cache(cfg.FEAT_CACHE)

In [ ]:
# ── Shared protocol (written by 4.0_cl_setup) ───────────────────────────────
# Every arm loads the SAME base head and the SAME hyperparameters, so lr/epochs/
# schedule are identical by construction even though the arms run in separate
# notebooks. The mechanism flags are the only difference.
P = json.load(open(f"{cfg.RESULTS_DIR}/stage1_protocol.json"))

FREEZE_LSTM = P["freeze_lstm"]
CEILING     = P["ceiling"]
CL_EPOCHS   = P["epochs"]
CL_LR       = P["lr"]
CL_WD       = P["wd"]
CL_BATCH    = P["batch_size"]
CL_LS       = P["label_smoothing"]

base_head = make_temporal_head(NUM_AFTER[0], freeze_lstm=FREEZE_LSTM, device=device)
base_head.load_state_dict(torch.load(f"{cfg.CKPT_DIR}/head_base.pt", map_location=device))
base_head.eval()

STREAM = dict(
    base_head=base_head, cache=CACHE, cl_tasks=CL_TASKS, task_classes=TASK_CLASSES,
    task_names=TASK_NAMES, num_after=NUM_AFTER, global_c2i=global_c2i,
    base_classes=cfg.SELECTED_CLASSES, device=device,
    epochs=CL_EPOCHS, lr=CL_LR, wd=CL_WD, batch_size=CL_BATCH,
    label_smoothing=CL_LS,
)

print(f"Protocol : {CL_EPOCHS} epochs, lr={CL_LR}, batch={CL_BATCH}, no best-val selection")
print(f"Boundary : LSTM {'FROZEN' if FREEZE_LSTM else 'TRAINABLE'}")
print(f"Ceiling  : {CEILING:.2%}" if CEILING else "Ceiling  : (none — run Section 3)")
print(f"Base head: {base_head.trainable_params():,} trainable, "
      f"base acc {eval_tasks_upto(base_head, CACHE, 0, device)[0]:.2%}")

## Run the arm

In [ ]:
res = run_cl_stream(arm="weight_align", use_weight_align=True, **STREAM)

R, m = save_arm_result(
    f"{cfg.RESULTS_DIR}/stage1_arm_weight_align.json",
    "WeightAlign", res["rows"], TASK_NAMES, res["bytes_per_class"],
    config={"mechanism": "weight_align"}, ceiling=CEILING,
)
cl_report(R, "WeightAlign", [TASK_NAMES[i] for i in [0] + CL_TASKS])
torch.save(res["head"].state_dict(), f"{cfg.CKPT_DIR}/head_weight_align.pt")

---

**Read the `weight_align: base X% -> Y%` line.** That is the correction happening. If
X == Y there was no norm imbalance to fix, which usually means the arm barely trained.